# Database Initialization Script

## What This Script Does

Creates a SQLite database (`churn.db`) with three tables:

### 1. `raw_data`
- Stores all 7,043 rows from `telco_churn.csv`
- Contains original customer data including Churn labels
- Used as the historical training dataset

### 2. `new_data`
- Takes 5 random rows from the original dataset
- Adds columns for predictions (`prediction`, `probability`, `churn_label`, `timestamp`)
- Used to simulate new customer predictions awaiting labeling

### 3. `api_keys`
- Stores API key hashes for authentication
- Currently has placeholder values (`placeholder_user_hash`, `placeholder_admin_hash`)
- Must be updated on the server with real hashed keys

## Why We Need This

- **Separation of concerns**: Raw data, new predictions, and keys are in different tables
- **Simulates production**: New customers go to `new_data`, get predictions, then can be labeled
- **Security**: Keys are hashed before storage (placeholders will be replaced)

## How to Use

1. Run this script once to create the database
2. On the server, run `update_keys.py` to enter your real API keys
3. The FastAPI app will use this database for all operations

## Output

- Database location: `app/data/DB/churn.db`
- 3 tables created with appropriate schemas
- Ready for use with the Churn Prediction API

In [2]:
"""
Simple script to create SQLite database with 3 tables:
1. raw_data - original telco churn data
2. new_data - 5 random rows for predictions
3. api_keys - API keys (simple placeholders)

Run this script ONCE to initialize the database.
"""

import sqlite3
import pandas as pd
from pathlib import Path

# Paths
DB_PATH = Path("../app/data/DB/churn.db")
CSV_PATH = Path("../data/raw/telco_churn.csv")


def create_database():
    """Create SQLite database with all three tables."""
    
    # Create data directory if it doesn't exist
    DB_PATH.parent.mkdir(parents=True, exist_ok=True)
    
    # Connect to database (creates file if not exists)
    conn = sqlite3.connect(str(DB_PATH))
    cursor = conn.cursor()
    
    # =========================================================
    # TABLE 1: raw_data (all original data)
    # =========================================================
    print("Loading CSV file...")
    df = pd.read_csv(CSV_PATH)
    print(f"Loaded {len(df)} rows from {CSV_PATH}")
    
    # Create table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS raw_data (
            customerID TEXT PRIMARY KEY,
            gender TEXT,
            SeniorCitizen INTEGER,
            Partner TEXT,
            Dependents TEXT,
            tenure INTEGER,
            PhoneService TEXT,
            MultipleLines TEXT,
            InternetService TEXT,
            OnlineSecurity TEXT,
            OnlineBackup TEXT,
            DeviceProtection TEXT,
            TechSupport TEXT,
            StreamingTV TEXT,
            StreamingMovies TEXT,
            Contract TEXT,
            PaperlessBilling TEXT,
            PaymentMethod TEXT,
            MonthlyCharges REAL,
            TotalCharges REAL,
            Churn TEXT
        )
    ''')
    
    # Insert all data
    df.to_sql('raw_data', conn, if_exists='replace', index=False)
    print(f" Table 'raw_data' created with {len(df)} rows")
    
    # =========================================================
    # TABLE 2: new_data (5 random rows for predictions)
    # =========================================================
    print("\nCreating 'new_data' table...")
    
    # Take 5 random rows
    df_new = df.sample(n=5, random_state=42).copy()
    
    # Add columns for predictions
    df_new['prediction'] = None
    df_new['probability'] = None
    df_new['churn_label'] = None
    df_new['timestamp'] = None
    
    # Create table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS new_data (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            customerID TEXT,
            gender TEXT,
            SeniorCitizen INTEGER,
            Partner TEXT,
            Dependents TEXT,
            tenure INTEGER,
            PhoneService TEXT,
            MultipleLines TEXT,
            InternetService TEXT,
            OnlineSecurity TEXT,
            OnlineBackup TEXT,
            DeviceProtection TEXT,
            TechSupport TEXT,
            StreamingTV TEXT,
            StreamingMovies TEXT,
            Contract TEXT,
            PaperlessBilling TEXT,
            PaymentMethod TEXT,
            MonthlyCharges REAL,
            TotalCharges REAL,
            Churn TEXT,
            prediction INTEGER,
            probability REAL,
            churn_label TEXT,
            timestamp TEXT
        )
    ''')
    
    # Insert 5 random rows
    df_new.to_sql('new_data', conn, if_exists='replace', index=False)
    print(f"Table 'new_data' created with {len(df_new)} rows")
    
    # Show which customers were added
    print("\n   Customers in new_data:")
    for _, row in df_new.iterrows():
        print(f"     - {row['customerID']}")
    
    # =========================================================
    # TABLE 3: api_keys (simple placeholders - you will replace)
    # =========================================================
    print("\nCreating 'api_keys' table...")
    
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS api_keys (
            key_hash TEXT PRIMARY KEY,
            name TEXT,
            role TEXT,
            rate_limit INTEGER DEFAULT 100,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    ''')
    
    # Simple placeholders (you will replace these manually on server)
    # Just put simple values for now
    cursor.execute('''
        INSERT OR REPLACE INTO api_keys (key_hash, name, role, rate_limit)
        VALUES (?, ?, ?, ?)
    ''', ("placeholder_user_hash", "default_user", "user", 100))
    
    cursor.execute('''
        INSERT OR REPLACE INTO api_keys (key_hash, name, role, rate_limit)
        VALUES (?, ?, ?, ?)
    ''', ("placeholder_admin_hash", "admin", "admin", 1000))
    
    print("Table 'api_keys' created with 2 placeholder keys")
    print(" IMPORTANT: Replace these with real hashed keys on server!")
    
    # Commit and close
    conn.commit()
    conn.close()
    
    # Summary
    print("\n" + "=" * 50)
    print(" DATABASE CREATED SUCCESSFULLY!")
    print("=" * 50)
    print(f"Database location: {DB_PATH}")
    print(f"File size: {DB_PATH.stat().st_size / 1024:.1f} KB")
  


if __name__ == "__main__":
    create_database()

Loading CSV file...
Loaded 7043 rows from ..\data\raw\telco_churn.csv
 Table 'raw_data' created with 7043 rows

Creating 'new_data' table...
Table 'new_data' created with 5 rows

   Customers in new_data:
     - 1024-GUALD
     - 0484-JPBRU
     - 3620-EHIMZ
     - 6910-HADCM
     - 8587-XYZSF

Creating 'api_keys' table...
Table 'api_keys' created with 2 placeholder keys
 IMPORTANT: Replace these with real hashed keys on server!

 DATABASE CREATED SUCCESSFULLY!
Database location: ..\app\data\DB\churn.db
File size: 1048.0 KB
